# 05 – Proportional Control

**Manufacturing context:** A position controller on a CNC axis. Open-loop commands give unpredictable results; closing the loop with feedback and a proportional gain `Kp` dramatically improves tracking.

**Control law:** &ensp; `u = Kp * (ref - y)`  
**Key tradeoff:** Higher `Kp` = faster response but more overshoot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
# Plant (second-order mass-spring-damper)
m = 1.0      # Mass [kg]
c = 1.0      # Damping [N*s/m]
k = 4.0      # Stiffness [N/m]
ref = 1.0    # Desired position setpoint [m]
t_end = 8.0
dt = 0.001

# Proportional gains to compare
Kp_values = [5.0, 20.0, 80.0]
# -----------------------------

In [ ]:
t = np.arange(0, t_end, dt)

def simulate_p_control(Kp):
    """Closed-loop proportional control: u = Kp * error."""
    x = np.zeros_like(t)
    v = np.zeros_like(t)
    u = np.zeros_like(t)
    for i in range(1, len(t)):
        e = ref - x[i-1]
        u[i] = Kp * e
        a = (u[i] - c * v[i-1] - k * x[i-1]) / m
        v[i] = v[i-1] + a * dt
        x[i] = x[i-1] + v[i] * dt
    return x, u

# Run all Kp cases
results = {}
efforts = {}
for Kp in Kp_values:
    x, u = simulate_p_control(Kp)
    results[f"Kp={Kp}"] = x
    efforts[f"Kp={Kp}"] = u

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

for label, x in results.items():
    axes[0].plot(t, x, label=label)
axes[0].axhline(ref, color="k", linestyle="--", linewidth=0.8, label="Reference")
axes[0].set_ylabel("Position [m]")
axes[0].set_title("Proportional Gain Sweep - Speed vs Overshoot")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

for label, u in efforts.items():
    axes[1].plot(t, u, label=label)
axes[1].set_ylabel("Control effort [N]")
axes[1].set_xlabel("Time [s]")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Student Exercise

1. Add `Kp=2.0` to the list. What happens to steady-state error?
2. Try to find a `Kp` that gives < 10% overshoot and < 0.5s rise time.
3. Why can't P-only control eliminate steady-state error on this plant?